# Silver Layer DLT Pipeline - Virtue Foundation Ghana

**Purpose:** Transform Bronze data into clean, structured Silver layer using Lakeflow Spark Declarative Pipelines (DLT)

**Transformations:**
* Parse JSON-stringified arrays into proper ArrayType columns
* Normalize null values (empty strings → null, empty arrays → null)
* Add computed columns (completeness_score, has_procedures, specialty_count, etc.)
* Apply data quality expectations
* Partition by region for query optimization

**Input:** `virtue_foundation.ghana.facilities_bronze`

**Output:** `virtue_foundation.ghana.facilities_silver`

**Data Quality:**
* Expect: name IS NOT NULL (drop violations)
* Expect: address_country = 'Ghana' (drop violations)
* Expect: address_city IS NOT NULL (log violations, keep records)

---

**Note:** This notebook should be run as a DLT pipeline, not as a standard notebook.

## 1. Setup and Configuration

Import DLT library and helper functions.

In [0]:
# DLT module is only available when running as a DLT pipeline
try:
    import dlt
    DLT_AVAILABLE = True
except ModuleNotFoundError:
    DLT_AVAILABLE = False
    print("⚠️  DLT module not available")
    print("This notebook is designed to run as a Delta Live Tables pipeline.")
    print("\nTo deploy as a DLT pipeline:")
    print("1. Go to Workflows → Delta Live Tables → Create Pipeline")
    print("2. Select this notebook: 02_Silver_Transformation_DLT")
    print("3. Set Target: virtue_foundation.ghana")
    print("4. Set Storage: /pipelines/virtue_foundation_ghana")
    print("5. Click Create → Start")
    print("\nRunning individual cells will not work for DLT notebooks.")

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, IntegerType, BooleanType, DoubleType
import json

# Configuration
SOURCE_TABLE = "virtue_foundation.ghana.facilities_bronze"
TARGET_TABLE = "facilities_silver"

if DLT_AVAILABLE:
    print(f"DLT Pipeline Configuration:")
    print(f"Source: {SOURCE_TABLE}")
    print(f"Target: {TARGET_TABLE}")
    print(f"Target Schema: virtue_foundation.ghana")

## 2. Helper Functions

Utility functions for JSON parsing and null normalization.

In [0]:
# Helper function to parse JSON arrays
def parse_json_array(json_str):
    """
    Parse JSON array string to Python list.
    Returns None for empty arrays, invalid JSON, or null values.
    """
    if json_str is None or json_str == "" or json_str == "[]":
        return None
    try:
        parsed = json.loads(json_str)
        if isinstance(parsed, list) and len(parsed) > 0:
            return parsed
        return None
    except:
        return None

# Register UDF
parse_json_array_udf = F.udf(parse_json_array, ArrayType(StringType()))

print("✅ Helper functions defined")

## 3. Bronze Source Definition

Define Bronze table as streaming source for incremental processing.

In [0]:
@dlt.table(
    name="facilities_bronze_source",
    comment="Bronze source for incremental processing"
)
def bronze_source():
    """
    Read from Bronze table as streaming source.
    This enables incremental processing in DLT.
    """
    return spark.readStream.table("virtue_foundation.ghana.facilities_bronze")

## 4. Silver Table with Full Transformations

Apply all transformations:
* JSON parsing
* Null normalization  
* Computed columns
* Data quality expectations

In [0]:
@dlt.table(
    name="facilities_silver",
    comment="""Cleaned and enriched facility data with:
    - Parsed JSON arrays (specialties, procedure, equipment, capability)
    - Normalized nulls
    - Computed quality metrics (completeness_score, has_*, specialty_count)
    - Data quality validations applied
    - Partitioned by address_stateOrRegion
    """,
    partition_cols=["address_stateOrRegion"],
    table_properties={
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true",
        "pipelines.autoOptimize.managed": "true",
        "quality": "silver",
        "layer": "curated"
    }
)
@dlt.expect_or_drop("valid_name", "name IS NOT NULL AND name != ''")
@dlt.expect_or_drop("valid_country", "address_country = 'Ghana' OR address_country IS NULL")
@dlt.expect("has_city", "address_city IS NOT NULL")
@dlt.expect("has_region", "address_stateOrRegion IS NOT NULL")
def facilities_silver():
    """
    Transform Bronze to Silver with comprehensive data quality improvements.
    """
    df = dlt.read_stream("facilities_bronze_source")
    
    # ========================================
    # 1. PARSE JSON ARRAY COLUMNS
    # ========================================
    json_cols = ['specialties', 'procedure', 'equipment', 'capability', 
                 'phone_numbers', 'websites', 'affiliationTypeIds']
    
    for col in json_cols:
        if col in df.columns:
            df = df.withColumn(
                col,
                F.when(
                    (F.col(col).isNotNull()) & 
                    (F.col(col) != "") & 
                    (F.col(col) != "[]"),
                    parse_json_array_udf(F.col(col))
                ).otherwise(None)
            )
    
    # ========================================
    # 2. NORMALIZE NULL VALUES
    # ========================================
    # Convert empty strings to null for string columns only (after JSON parsing)
    for field in df.schema.fields:
        col_name = field.name
        # Check current schema type, not original
        current_type = df.schema[col_name].dataType
        if isinstance(current_type, StringType):
            df = df.withColumn(
                col_name,
                F.when((F.col(col_name) == "") | (F.col(col_name) == "null"), None)
                .otherwise(F.trim(F.col(col_name)))
            )
    
    # ========================================
    # 3. ADD COMPUTED BOOLEAN FLAGS
    # ========================================
    df = df.withColumn(
        "has_procedures",
        F.when(
            F.col("procedure").isNotNull() & (F.size(F.col("procedure")) > 0),
            True
        ).otherwise(False)
    )
    
    df = df.withColumn(
        "has_equipment",
        F.when(
            F.col("equipment").isNotNull() & (F.size(F.col("equipment")) > 0),
            True
        ).otherwise(False)
    )
    
    df = df.withColumn(
        "has_capability",
        F.when(
            F.col("capability").isNotNull() & (F.size(F.col("capability")) > 0),
            True
        ).otherwise(False)
    )
    
    df = df.withColumn(
        "has_specialties",
        F.when(
            F.col("specialties").isNotNull() & (F.size(F.col("specialties")) > 0),
            True
        ).otherwise(False)
    )
    
    # ========================================
    # 4. ADD COUNT COLUMNS
    # ========================================
    df = df.withColumn(
        "specialty_count",
        F.when(F.col("specialties").isNotNull(), F.size(F.col("specialties"))).otherwise(0)
    )
    
    df = df.withColumn(
        "procedure_count",
        F.when(F.col("procedure").isNotNull(), F.size(F.col("procedure"))).otherwise(0)
    )
    
    df = df.withColumn(
        "equipment_count",
        F.when(F.col("equipment").isNotNull(), F.size(F.col("equipment"))).otherwise(0)
    )
    
    # ========================================
    # 5. ADD COMPLETENESS SCORE
    # ========================================
    # Define critical fields for completeness calculation
    # Separate array fields from string fields
    array_fields = ['specialties', 'phone_numbers']
    string_fields = ['name', 'address_city', 'address_stateOrRegion', 'facilityTypeId',
                     'numberDoctors', 'capacity', 'description', 'email']
    
    # Count non-null string fields
    string_checks = [
        F.when(F.col(field).isNotNull() & (F.col(field) != ""), 1).otherwise(0)
        for field in string_fields if field in df.columns
    ]
    
    # Count non-null array fields (check size > 0)
    array_checks = [
        F.when(F.col(field).isNotNull() & (F.size(F.col(field)) > 0), 1).otherwise(0)
        for field in array_fields if field in df.columns
    ]
    
    # Combine all checks
    non_null_count = sum(string_checks + array_checks)
    total_critical_fields = len([f for f in string_fields if f in df.columns]) + \
                           len([f for f in array_fields if f in df.columns])
    
    df = df.withColumn(
        "completeness_score",
        (non_null_count / total_critical_fields).cast(DoubleType())
    )
    
    # ========================================
    # 6. ADD CONTACT FLAGS
    # ========================================
    df = df.withColumn(
        "has_phone",
        F.when(
            F.col("phone_numbers").isNotNull() & (F.size(F.col("phone_numbers")) > 0),
            True
        ).otherwise(False)
    )
    
    df = df.withColumn(
        "has_email",
        F.col("email").isNotNull()
    )
    
    df = df.withColumn(
        "has_website",
        F.when(
            F.col("websites").isNotNull() & (F.size(F.col("websites")) > 0),
            True
        ).otherwise(F.col("officialWebsite").isNotNull())
    )
    
    df = df.withColumn(
        "has_any_contact",
        F.col("has_phone") | F.col("has_email") | F.col("has_website")
    )
    
    # ========================================
    # 7. CAST NUMERIC FIELDS
    # ========================================
    if 'numberDoctors' in df.columns:
        df = df.withColumn(
            "numberDoctors",
            F.col("numberDoctors").cast(IntegerType())
        )
    
    if 'capacity' in df.columns:
        df = df.withColumn(
            "capacity",
            F.col("capacity").cast(IntegerType())
        )
    
    if 'yearEstablished' in df.columns:
        df = df.withColumn(
            "yearEstablished",
            F.col("yearEstablished").cast(IntegerType())
        )
    
    # ========================================
    # 8. ADD PROCESSING TIMESTAMP
    # ========================================
    df = df.withColumn(
        "silver_processed_timestamp",
        F.current_timestamp()
    )
    
    return df

## 5. Data Quality Summary View

Create a view for monitoring data quality metrics.

In [0]:
@dlt.view(
    name="silver_quality_metrics",
    comment="Data quality metrics for Silver layer monitoring"
)
def silver_quality_metrics():
    """
    Aggregate quality metrics for monitoring.
    """
    df = dlt.read("facilities_silver")
    
    return df.agg(
        F.count("*").alias("total_records"),
        F.avg("completeness_score").alias("avg_completeness_score"),
        F.sum(F.when(F.col("has_procedures"), 1).otherwise(0)).alias("records_with_procedures"),
        F.sum(F.when(F.col("has_equipment"), 1).otherwise(0)).alias("records_with_equipment"),
        F.sum(F.when(F.col("has_capability"), 1).otherwise(0)).alias("records_with_capability"),
        F.sum(F.when(F.col("has_specialties"), 1).otherwise(0)).alias("records_with_specialties"),
        F.sum(F.when(F.col("has_any_contact"), 1).otherwise(0)).alias("records_with_contact"),
        F.sum(F.when(F.col("completeness_score") < 0.3, 1).otherwise(0)).alias("low_quality_records"),
        F.count_distinct("address_stateOrRegion").alias("unique_regions"),
        F.count_distinct("facilityTypeId").alias("unique_facility_types")
    )

## ✅ Silver DLT Pipeline Complete!

**What's been created:**
* ✅ DLT streaming source from Bronze table
* ✅ Silver table with comprehensive transformations
* ✅ JSON array parsing (specialties, procedure, equipment, capability, etc.)
* ✅ Null normalization across all columns
* ✅ Computed columns: has_*, *_count, completeness_score
* ✅ Contact information flags
* ✅ Data quality expectations (drop invalid names/countries)
* ✅ Quality metrics view for monitoring
* ✅ Partitioning by address_stateOrRegion

**Data Quality Expectations:**
* **Drop records:** name IS NULL or address_country ≠ 'Ghana'
* **Log warnings:** missing city or region

**Computed Metrics:**
* `completeness_score`: Percentage of critical fields populated (0.0 - 1.0)
* `has_procedures`, `has_equipment`, `has_capability`, `has_specialties`: Boolean flags
* `specialty_count`, `procedure_count`, `equipment_count`: Array size counts
* `has_phone`, `has_email`, `has_website`, `has_any_contact`: Contact flags

**Next Steps:**
1. **Deploy this pipeline:**
   * Go to **Workflows → Delta Live Tables**
   * Click **Create Pipeline**
   * Select this notebook
   * Target: `virtue_foundation.ghana`
   * Storage location: `/pipelines/virtue_foundation_ghana`
   * Click **Create** then **Start**

2. **Verify output:**
   ```sql
   SELECT COUNT(*) FROM virtue_foundation.ghana.facilities_silver;
   SELECT * FROM virtue_foundation.ghana.silver_quality_metrics;
   ```

3. **Run Gold aggregations:** `03_Gold_Regional_Summary`

**Pipeline Configuration:**
```json
{
  "name": "virtue_foundation_ghana_silver",
  "target": "virtue_foundation.ghana",
  "storage": "/pipelines/virtue_foundation_ghana",
  "configuration": {
    "pipelines.applyChangesPreviewEnabled": "true"
  },
  "clusters": [
    {
      "label": "default",
      "autoscale": {
        "min_workers": 1,
        "max_workers": 5
      }
    }
  ]
}
```